# Building an AI Project Manager Agent with LangGraph

In this notebook you build an agent that takes a project description and a team, then plans the work: it breaks the project into tasks, maps their dependencies, schedules them, assigns people, and audits the plan for risk. A Groq-hosted LLM does the reasoning and LangGraph runs the steps in order.

## Learning Objectives

At the end of this notebook, you should be able to:

- Define Pydantic models that validate and repair structured LLM output.
- Build a multi-step agent as a LangGraph state graph with shared state.
- Implement a feedback loop that sends the plan back for revision based on a risk score.
- Visualise a generated project schedule as a Gantt chart with Plotly.

### What These Objectives Mean for You as an AI Product Manager

You will not ship this code, but you will specify, scope, and review AI features that work like it. Each objective gives you a skill for those conversations with engineers and stakeholders:

| Objective | Why it matters in your role |
|---|---|
| Define Pydantic models (Pydantic is a Python library that describes the shape of data and checks it) | Writing down exactly what a valid AI answer looks like is the same skill as writing acceptance criteria. It tells you what "working" means for an AI feature. |
| Build a multi-step agent as a LangGraph state graph | You can read an agent's design as steps, hand-offs, and shared memory. This helps you scope features, estimate effort, and decide where a check or a human review belongs. |
| Implement a feedback loop driven by a risk score | You see how quality gates and stopping rules work, and why every extra pass trades quality against cost and waiting time. |
| Visualise the schedule as a Gantt chart | An agent is only useful if its output reaches people in a form they can act on. This is the step that turns raw results into a stakeholder-ready view. |

## Why Build the Agent as a Graph?

A single prompt such as "plan this project" produces one long, hard-to-check answer. Splitting the work into small steps gives each step a narrow job, a clear input and output, and its own validation. If the schedule is poor, you can fix the scheduler without touching the rest.

LangGraph models this as a **graph**: each step is a node, and edges say which step runs next. The alternative is a plain Python function that calls the steps in sequence. That works for a straight line, but a graph makes the **loop** in this agent explicit: when the risk score is high, the plan goes back to the scheduler for another attempt.

```mermaid
flowchart TD
    A[Scoper: split into tasks] --> B[Mapper: find dependencies]
    B --> C[Scheduler: set start and end days]
    C --> D[Allocator: assign people]
    D --> E[Auditor: score risk]
    E -->|risk high and iterations left| F[Optimizer: suggest a change]
    F --> C
    E -->|risk low or limit reached| G[End]
```

## Imports & Setup

The agent talks to the model through `ChatGroq`, which reads your `GROQ_API_KEY` from the `.env` file via `load_dotenv()`. The `temperature` is set low (`0.1`) because every node asks for structured JSON, and lower randomness gives more consistent output. The alternative is to set the key directly in `os.environ`, but a `.env` file keeps secrets out of the notebook.

> **Note:** On Groq's free tier, `openai/gpt-oss-20b` is limited to 8,000 tokens per minute (see the [rate limits](https://console.groq.com/docs/rate-limits)). A run can stop with a `429` rate limit error. Wait a few seconds and run the cell again.

In [1]:
import uuid
from datetime import datetime, timedelta

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_groq import ChatGroq

# Load Env
load_dotenv()
# Ensure you have set GROQ_API_KEY in your .env file or environment
# os.environ['GROQ_API_KEY'] = "your_key_here"

llm = ChatGroq(
    model="openai/gpt-oss-20b", temperature=0.1
)  # Recommended model for logic

## Unified Data Models

### What Is Pydantic?

**Pydantic** is a Python library for describing what a piece of data should look like and then checking real data against that description.

You write the description as a **model**: a list of fields and what each must contain. For example, a `Task` must have a name (text) and an estimate in days (a whole number). When data arrives, Pydantic either accepts it, converts it where that is safe (the text `"3"` becomes the number `3`), or rejects it with a clear error that names the field that is wrong.

For a product manager, think of it as three familiar things at once:

- **A form with required fields.** A submission with a missing or wrong field is refused, not silently accepted.
- **An API contract.** Both sides agree on the shape of the data, so a mismatch is caught at the boundary.
- **Acceptance criteria for an AI answer.** "A valid answer has a task name and a number of days" is a testable rule, which an open-ended text reply is not.

This matters for AI features because an LLM answers in free text. The rest of the system needs predictable, structured data, so Pydantic sits between the two and decides whether the model's answer is usable.

### Why Validate LLM Output?

An LLM returns text, not Python objects. To use its answer safely, each node asks for JSON and **validates** it against a Pydantic model. If the JSON does not fit, you find out immediately instead of failing later in the chart code.

Two techniques make the models forgiving, because the LLM does not always use the exact field names you asked for:

- **`AliasChoices`** lets one field accept several names, for example `title`, `name`, or `task_name` for a task's name.
- **`model_validator(mode="before")`** reshapes the raw data before validation, for example wrapping a bare list in the expected dictionary.

The alternative is a strict schema that rejects anything unexpected. That is simpler, but it fails more often with a model that varies its output.

In [2]:
from typing import Annotated, Any

from pydantic import AliasChoices, BaseModel, BeforeValidator, Field, model_validator


# --- Helpers ---
def force_string(v):
    return str(v) if v is not None else None


StringId = Annotated[str | None, BeforeValidator(force_string)]


# --- Core Entities ---
class Task(BaseModel):
    id: StringId = None
    task_name: str = Field(
        ..., validation_alias=AliasChoices("title", "name", "task_name", "task")
    )
    task_description: str = Field(
        default="",
        validation_alias=AliasChoices("description", "task_description", "desc"),
    )
    estimated_day: int = Field(
        ...,
        validation_alias=AliasChoices(
            "duration_days",
            "estimated_days",
            "estimated_day",
            "duration",
            "estimated_time",
            "time",
        ),
    )
    required_skill: str = Field(
        default="General",
        validation_alias=AliasChoices("required_skill", "skill", "skills"),
    )


class TeamMember(BaseModel):
    name: str
    role: str
    skills: list[str]
    seniority: str


class Team(BaseModel):
    team_members: list[TeamMember]


# --- Workflow Containers (With Smart Auto-Fixers) ---


class TaskList(BaseModel):
    task: list[Task] = Field(..., validation_alias=AliasChoices("tasks", "task"))

    @model_validator(mode="before")
    @classmethod
    def wrap_list(cls, data: Any) -> Any:
        if isinstance(data, list):
            return {"task": data}
        if isinstance(data, dict) and "tasks" in data:
            return {"task": data["tasks"]}
        return data


class Dependency(BaseModel):
    task_id: str = Field(..., validation_alias=AliasChoices("task_id", "id", "task"))
    dependent_on: list[str] = Field(
        ...,
        validation_alias=AliasChoices(
            "dependent_on", "dependencies", "deps", "depends_on"
        ),
    )


class DependencyList(BaseModel):
    dependencies: list[Dependency] = Field(
        ..., validation_alias=AliasChoices("dependencies", "deps")
    )

    @model_validator(mode="before")
    @classmethod
    def wrap_list(cls, data: Any) -> Any:
        # Case 1: Raw List
        if isinstance(data, list):
            return {"dependencies": data}

        # Case 2: Dictionary Wrapper
        if isinstance(data, dict):
            # Check if it has the key "dependencies"
            deps = data.get("dependencies", data.get("deps"))

            # If the value inside is a Dict {"A": ["B"], "C": []}, convert to List format
            if isinstance(deps, dict):
                converted = [{"task_id": k, "dependent_on": v} for k, v in deps.items()]
                return {"dependencies": converted}

            # If the value inside is already a list, just return properly
            if isinstance(deps, list):
                return {"dependencies": deps}

        return data


class TaskSchedule(BaseModel):
    task: Task
    start_day: int
    end_day: int


class Schedule(BaseModel):
    schedule: list[TaskSchedule]


class TaskAllocation(BaseModel):
    task: Task
    team_member: TeamMember


class TaskAllocationList(BaseModel):
    task_allocations: list[TaskAllocation]

    @model_validator(mode="before")
    @classmethod
    def wrap_list(cls, data: Any) -> Any:
        if isinstance(data, list):
            return {"task_allocations": data}

        if isinstance(data, dict):
            # Check for common keys
            payload = (
                data.get("task_allocations")
                or data.get("allocs")
                or data.get("allocations")
            )

            # If payload is a Dict {"Task A": "Alice"}, convert to List format
            if isinstance(payload, dict):
                converted = [
                    {"task_id": k, "member_name": v} for k, v in payload.items()
                ]
                return {"task_allocations": converted}

            if isinstance(payload, list):
                return {"task_allocations": payload}

        return data


class Risk(BaseModel):
    task_name: str
    score: int
    reason: str


class RiskList(BaseModel):
    risks: list[Risk]

## Agent State Definition

The **state** is the shared memory of the agent. Every node reads from it and returns only the keys it wants to update, and LangGraph merges those updates into the state. This means nodes never call each other directly: the scoper writes `tasks`, the mapper reads `tasks` and writes `dependencies`, and so on.

The state also carries the loop controls: `iteration_number`, `max_iteration`, and the list of risk scores from each pass. The alternative is passing arguments between functions, but a shared state makes it easy to add a step or inspect what happened.

In [3]:
from typing import TypedDict


class AgentState(TypedDict):
    project_description: str
    team: Team
    tasks: TaskList
    dependencies: list[dict]
    schedule: Schedule
    task_allocations: TaskAllocationList
    risks: RiskList
    iteration_number: int
    max_iteration: int
    insights: list[str]
    project_risk_score_iterations: list[int]

## Node Logic

Five of the six nodes call the LLM through structured output:

| Node | Reads | Writes | Job |
|---|---|---|---|
| `scope_decomposition_node` | project, team skills | `tasks` | Split the project into tasks of at most 4 days. |
| `dependency_mapping_node` | `tasks` | `dependencies` | Say which tasks must finish first. |
| `smart_scheduler_node` | `tasks`, `dependencies`, `insights` | `schedule` | Give each task a start and end day. |
| `resource_allocation_node` | `tasks`, `team` | `task_allocations` | Match tasks to people by skill. |
| `risk_audit_node` | `schedule`, `task_allocations` | `risks`, score history | Score each task from 0 to 10 and add up the total. |

The sixth node, `optimization_insight_node`, asks the LLM for one concrete change to lower the risk and stores it in `insights`. The scheduler reads those insights on its next pass, which is how the feedback loop improves the plan.

Each node uses `with_structured_output(..., method="json_mode")`, which asks the model for JSON and parses it into the Pydantic model. The alternative is function calling, which constrains the output more tightly but is not supported equally well by every model.

> **Note:** The validators absorb many shapes of JSON, but not every one. Occasionally the model returns JSON the validators cannot use (an `OutputParserException`), or Groq rejects an empty reply (a `400` error with `json_validate_failed`). The output is random, so each structured-output call uses `.with_retry(stop_after_attempt=3)`, which asks the model again if a call fails. If a node still fails after three attempts, wait a moment and run the cell again.

In [4]:
# --- Node Functions ---


def scope_decomposition_node(state: AgentState):
    """Decomposes project into granular tasks."""
    print("--- Node: Scoper ---")
    prompt = f"""
    Project: {state["project_description"]}
    Team Skills Available: {[m.skills for m in state["team"].team_members]}
    
    Break this project into granular tasks (max 4 days per task).
    
    CRITICAL INSTRUCTIONS:
    1. Return a JSON object matching the TaskList schema.
    2. REQUIRED FIELDS: 'task_name', 'task_description', 'estimated_day', 'required_skill'.
    3. Do NOT assign names (like Alice or Bob). Only identify the 'required_skill' (e.g., Python, React, Testing).
    
    Example Output:
    {{
      "tasks": [
        {{
            "task_name": "Setup Repo", 
            "task_description": "Initialize Git repo and CI/CD pipelines", 
            "estimated_day": 1, 
            "required_skill": "DevOps"
        }}
      ]
    }}
    """
    struct_llm = llm.with_structured_output(TaskList, method="json_mode").with_retry(
        stop_after_attempt=3
    )
    response = struct_llm.invoke(prompt)

    # Generate IDs if missing
    for t in response.task:
        if not t.id:
            t.id = str(uuid.uuid4())[:4]

    return {"tasks": response}


def dependency_mapping_node(state: AgentState):
    """Identifies dependencies between tasks."""
    print("--- Node: Mapper ---")
    tasks_fmt = "\n".join(
        [f"ID: {t.id} | Name: {t.task_name}" for t in state["tasks"].task]
    )

    prompt = f"""
    Map dependencies for these tasks:
    {tasks_fmt}
    
    Return a JSON object matching DependencyList. 
    Use IDs to reference tasks. 
    
    Example Output:
    {{
      "dependencies": [
         {{"task_id": "1a2b", "dependent_on": []}},
         {{"task_id": "3c4d", "dependent_on": ["1a2b"]}}
      ]
    }}
    """
    struct_llm = llm.with_structured_output(
        DependencyList, method="json_mode"
    ).with_retry(stop_after_attempt=3)
    response = struct_llm.invoke(prompt)
    return {"dependencies": response.dependencies}


def smart_scheduler_node(state: AgentState):
    """Schedules tasks (Day 0, Day 1...)"""
    print("--- Node: Scheduler ---")

    class SimpleSchedItem(BaseModel):
        task_id: str = Field(
            ..., validation_alias=AliasChoices("task_id", "id", "task_name", "task")
        )
        start: int = Field(..., validation_alias=AliasChoices("start", "start_day"))
        end: int = Field(..., validation_alias=AliasChoices("end", "end_day"))

    class SimpleSched(BaseModel):
        items: list[SimpleSchedItem]

        @model_validator(mode="before")
        @classmethod
        def wrap(cls, data):
            if isinstance(data, list):
                return {"items": data}
            if isinstance(data, dict):
                if "tasks" in data:
                    return {"items": data["tasks"]}
                if "schedule" in data:
                    return {"items": data["schedule"]}
                if "timeline" in data:
                    return {"items": data["timeline"]}
                # Dict keyed by task id: {"1a2b": {"start": 1, "end": 2}}
                if data and all(isinstance(v, dict) for v in data.values()):
                    return {"items": [{"task_id": k, **v} for k, v in data.items()]}
            return data

    prompt = f"""
    Schedule these tasks (Tasks: {state["tasks"]}) 
    considering Dependencies: {state.get("dependencies")}
    Previous Insights: {state.get("insights", [])}
    
    Return JSON with start/end days (integers).
    """
    struct_llm = llm.with_structured_output(SimpleSched, method="json_mode").with_retry(
        stop_after_attempt=3
    )
    resp = struct_llm.invoke(prompt)

    task_map = {t.id: t for t in state["tasks"].task}
    name_map = {t.task_name: t for t in state["tasks"].task}

    final_sched = []
    for item in resp.items:
        task = task_map.get(str(item.task_id)) or name_map.get(str(item.task_id))
        if task:
            final_sched.append(
                TaskSchedule(task=task, start_day=item.start, end_day=item.end)
            )

    return {"schedule": Schedule(schedule=final_sched)}


def resource_allocation_node(state: AgentState):
    """Allocates tasks to team members."""
    print("--- Node: Allocator ---")

    class SimpleAllocItem(BaseModel):
        task_id: str = Field(
            ..., validation_alias=AliasChoices("task_id", "id", "task_name", "task")
        )
        member_name: str = Field(
            ...,
            validation_alias=AliasChoices(
                "member_name", "member", "assignee", "person", "name"
            ),
        )

    class SimpleAlloc(BaseModel):
        allocs: list[SimpleAllocItem]

        @model_validator(mode="before")
        @classmethod
        def wrap(cls, data):
            if isinstance(data, list):
                return {"allocs": data}
            if isinstance(data, dict):
                if "allocations" in data:
                    return {"allocs": data["allocations"]}
                if "task_allocations" in data:
                    return {"allocs": data["task_allocations"]}
                if "assignments" in data:
                    return {"allocs": data["assignments"]}
                if "team" in data:
                    return {"allocs": data["team"]}
            return data

    prompt = f"""
    Allocate tasks: {state["tasks"]}
    To Team: {state["team"]}
    
    IMPORTANT: Return JSON.
    
    Output Format Example:
    {{
      "allocs": [
        {{"task_id": "1a2b", "member_name": "Alice"}},
        {{"task_id": "3c4d", "member_name": "Bob"}}
      ]
    }}
    """
    struct_llm = llm.with_structured_output(SimpleAlloc, method="json_mode").with_retry(
        stop_after_attempt=3
    )
    resp = struct_llm.invoke(prompt)

    task_map = {t.id: t for t in state["tasks"].task}
    name_map = {t.task_name: t for t in state["tasks"].task}
    member_map = {m.name: m for m in state["team"].team_members}

    final_allocs = []
    for a in resp.allocs:
        task = task_map.get(str(a.task_id)) or name_map.get(str(a.task_id))
        member = member_map.get(a.member_name)
        if task and member:
            final_allocs.append(TaskAllocation(task=task, team_member=member))

    return {"task_allocations": TaskAllocationList(task_allocations=final_allocs)}


def risk_audit_node(state: AgentState):
    """Calculates risk score."""
    print("--- Node: Auditor ---")

    # *** REPAIR FUNCTION: Fixes incomplete risk items before validation ***
    def repair_risk_item(item: Any) -> Any:
        if isinstance(item, dict):
            # If missing reason, fill it with a placeholder
            if "reason" not in item:
                # Try to find synonyms
                item["reason"] = item.get(
                    "description",
                    item.get(
                        "justification", item.get("explanation", "No reason provided")
                    ),
                )

            # If missing task_name, try to grab task_id
            if "task_name" not in item and "task_id" in item:
                item["task_name"] = str(item["task_id"])

            # Ensure score exists
            if "score" not in item:
                item["score"] = 5  # Default medium risk
        return item

    class SimpleRisk(BaseModel):
        task_name: str = Field(
            ...,
            validation_alias=AliasChoices(
                "task_name", "task", "name", "Task Name", "Task"
            ),
        )
        score: int = Field(
            ..., validation_alias=AliasChoices("score", "risk_score", "Risk Score")
        )
        reason: str = Field(
            ...,
            validation_alias=AliasChoices(
                "reason", "description", "justification", "Reason", "explanation"
            ),
        )

        @model_validator(mode="before")
        @classmethod
        def fix_data(cls, data):
            return repair_risk_item(data)

    class SimpleRiskList(BaseModel):
        risks: list[SimpleRisk]

        @model_validator(mode="before")
        @classmethod
        def wrap(cls, data):
            # 1. Unwrap common nested structures
            if isinstance(data, dict):
                # Search for the list key
                for key in [
                    "risks",
                    "issues",
                    "threats",
                    "audit",
                    "High-Risk Tasks",
                    "tasks",
                    "Risk Breakdown",
                    "recommendations",
                ]:
                    if key in data and isinstance(data[key], list):
                        return {"risks": data[key]}

                # Search values
                for val in data.values():
                    if (
                        isinstance(val, list)
                        and len(val) > 0
                        and isinstance(val[0], dict)
                    ):
                        return {"risks": val}

            if isinstance(data, list):
                return {"risks": data}
            return data

    prompt = f"""
    Audit this Plan for Risks (0-10 score):
    Schedule: {state.get("schedule")}
    Allocations: {state.get("task_allocations")}
    
    IMPORTANT: Return a JSON object with a single key "risks" containing a list of objects.
    Each object MUST have: "task_name", "score", and "reason".
    """
    struct_llm = llm.with_structured_output(
        SimpleRiskList, method="json_mode"
    ).with_retry(stop_after_attempt=3)
    resp = struct_llm.invoke(prompt)

    final_risks = [
        Risk(task_name=r.task_name, score=r.score, reason=r.reason) for r in resp.risks
    ]
    risks_obj = RiskList(risks=final_risks)

    score = sum(r.score for r in final_risks)
    return {
        "risks": risks_obj,
        "project_risk_score_iterations": state.get("project_risk_score_iterations", [])
        + [score],
        "iteration_number": state.get("iteration_number", 0) + 1,
    }


def optimization_insight_node(state: AgentState):
    """Generates advice if risk is high."""
    print("--- Node: Optimizer ---")
    prompt = f"Risks: {state['risks']}. Suggest 1 concrete schedule/allocation change to lower risk."
    insight = llm.invoke(prompt).content
    return {"insights": state.get("insights", []) + [insight]}

## Graph Structure

Here you wire the nodes together. `add_edge` creates a fixed transition, and `add_conditional_edges` calls `routing_logic` to decide the next step after the auditor:

- If `iteration_number` has reached `max_iteration`, or the latest risk score is below 15, the graph ends.
- Otherwise it goes to the `optimizer`, which loops back to the `scheduler`.

`MemorySaver` is a checkpointer: it records the state after every step, keyed by a `thread_id`. The alternative is a `while` loop in plain Python, but the graph gives you the checkpoints and a structure you can inspect.

In [5]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph


def routing_logic(state: AgentState):
    # Stop if we hit max iterations or risk is low (e.g. < 15)
    last_score = state["project_risk_score_iterations"][-1]
    if state["iteration_number"] >= state["max_iteration"] or last_score < 15:
        return END
    return "optimizer"


workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("scoper", scope_decomposition_node)
workflow.add_node("mapper", dependency_mapping_node)
workflow.add_node("scheduler", smart_scheduler_node)
workflow.add_node("allocator", resource_allocation_node)
workflow.add_node("auditor", risk_audit_node)
workflow.add_node("optimizer", optimization_insight_node)

# Add Edges
workflow.set_entry_point("scoper")
workflow.add_edge("scoper", "mapper")
workflow.add_edge("mapper", "scheduler")
workflow.add_edge("scheduler", "allocator")
workflow.add_edge("allocator", "auditor")
workflow.add_conditional_edges("auditor", routing_logic)
workflow.add_edge("optimizer", "scheduler")  # Loop back

graph = workflow.compile(checkpointer=MemorySaver())

## Execution

Running the agent needs an **initial state**: the project description, your team, and the loop limits. `graph.invoke` then runs the graph from the entry point until it reaches `END`. The `thread_id` tells the checkpointer which conversation this run belongs to.

Change the team or the description to see how the plan changes. Keep `max_iteration` small, because every pass makes several LLM calls and counts towards the rate limit.

In [6]:
# 1. Define Team
my_team = Team(
    team_members=[
        TeamMember(
            name="Alice",
            role="Lead Dev",
            skills=["Python", "LangGraph", "Architecture"],
            seniority="Senior",
        ),
        TeamMember(
            name="Bob", role="Frontend", skills=["React", "UI/UX"], seniority="Mid"
        ),
        TeamMember(
            name="Charlie", role="QA", skills=["Testing", "Python"], seniority="Junior"
        ),
    ]
)

# 2. Define Initial State
init_state = {
    "project_description": "Create a secure, multi-user AI dashboard using LangGraph and React.",
    "team": my_team,
    "iteration_number": 0,
    "max_iteration": 2,
    "insights": [],
    "project_risk_score_iterations": [],
}

# 3. Run
print("Starting Workflow...")
final_state = graph.invoke(init_state, {"configurable": {"thread_id": "v3_production"}})
print("Workflow Finished!")

Starting Workflow...
--- Node: Scoper ---


--- Node: Mapper ---


--- Node: Scheduler ---


--- Node: Allocator ---


--- Node: Auditor ---


--- Node: Optimizer ---


--- Node: Scheduler ---


--- Node: Allocator ---


--- Node: Auditor ---


Workflow Finished!


### Reading the Execution Log

The printed node names show the path the agent took. In a typical run it goes through the five nodes once (Scoper, Mapper, Scheduler, Allocator, Auditor), then the Optimizer, and then repeats Scheduler, Allocator, and Auditor before finishing. That second pass is the feedback loop: the first risk score was too high to stop, so the agent asked for a suggestion and re-planned.

The loop stops after two passes because `max_iteration` is set to `2`, even if the risk is still high.

## Visualization & Reporting

The final state holds Python objects, so this cell converts them into a table and a chart:

1. Build a lookup from task name to assignee. A dictionary lookup is much faster than searching the allocation list for every task.
2. Turn each task's start and end day into real dates, counting from today.
3. Put the rows in a pandas `DataFrame` and draw them with `px.timeline`, coloured by assignee.

A Gantt chart suits this data because it shows what runs when and who is responsible, which is hard to read from a table of numbers.

In [7]:
# --- 1. Report Metadata ---
display(Markdown(f"# Project Report: {final_state['project_description']}"))
display(
    Markdown(f"**Risk Score History:** {final_state['project_risk_score_iterations']}")
)

# --- 2. Org Chart ---
print("\nTeam Structure:")
for m in final_state["team"].team_members:
    print(f"└── {m.name} ({m.role})")

# --- 3. Gantt Chart Data Prep ---
sched_data = []
start_date_base = datetime.now()

# Create a lookup for Allocations to avoid O(N^2) loop
alloc_map = {
    a.task.task_name: a.team_member.name
    for a in final_state["task_allocations"].task_allocations
}

for item in final_state["schedule"].schedule:
    t_name = item.task.task_name
    assignee = alloc_map.get(t_name, "Unassigned")

    # Date Math
    s_date = start_date_base + timedelta(days=item.start_day)
    e_date = start_date_base + timedelta(days=item.end_day)

    sched_data.append(
        {
            "Task": t_name,
            "Start": s_date.strftime("%Y-%m-%d"),
            "Finish": e_date.strftime("%Y-%m-%d"),
            "Assignee": assignee,
            "Duration": item.end_day - item.start_day,
        }
    )

df = pd.DataFrame(sched_data).sort_values("Start")

# --- 4. Plot ---
if not df.empty:
    fig = px.timeline(
        df,
        x_start="Start",
        x_end="Finish",
        y="Task",
        color="Assignee",
        title="Project Schedule",
    )
    fig.update_yaxes(autorange="reversed")
    fig.show()
else:
    print("No schedule data to plot.")

# Project Report: Create a secure, multi-user AI dashboard using LangGraph and React.

**Risk Score History:** [52, 64]


Team Structure:
└── Alice (Lead Dev)
└── Bob (Frontend)
└── Charlie (QA)


### Reading the Results

The numbers below come from one run of this notebook. Yours will differ, because the LLM answers differently each time.

- **Risk score history `[33, 42]`.** The first pass scored 33 and the second scored 42, so the risk went **up** after the optimizer's suggestion. The agent does not guarantee an improvement: it only passes a suggestion to the scheduler. Both scores are far above the threshold of 15, so the loop ended because it hit `max_iteration`, not because the plan became safe.
- **Why the score is high.** The score is the **sum** of the per-task risk scores, so it grows with the number of tasks. This plan had 10 tasks, so even a modest score per task adds up to a large total. This means you should only compare scores between plans of similar size, and the threshold of 15 is hard to reach for a project of this scale.
- **Allocation.** Alice, the senior lead, received 6 of the 10 tasks, Bob received 2, and Charlie received 2. The allocator follows the skills in the prompt, so one person with broad skills can become a bottleneck. A sensible next step is to add a workload rule to the allocator prompt or the audit.
- **The chart.** Each colour is one person. Look for long stretches of a single colour: they show where one person carries the plan.

## Summary

In this notebook you:

- Defined Pydantic models with aliases and validators that validate and repair structured LLM output.
- Built a multi-step agent as a LangGraph state graph, with shared state and one node per planning step.
- Implemented a feedback loop that routes the plan to an optimizer and back to the scheduler, controlled by a risk threshold and an iteration limit.
- Visualised the generated schedule as a Gantt chart with Plotly, and read the risk history to judge the plan.

## References & Further Reading

- [**LangGraph Documentation**](https://docs.langchain.com/oss/python/langgraph/overview): Official guide to building stateful agents as graphs
- [**LangGraph Workflows and Agents**](https://docs.langchain.com/oss/python/langgraph/workflows-agents): Common patterns, including routing and feedback loops
- [**ChatGroq Integration**](https://docs.langchain.com/oss/python/integrations/chat/groq): How LangChain connects to Groq-hosted models
- [**Groq Rate Limits**](https://console.groq.com/docs/rate-limits): Why a run can stop with a `429` error and how the limits work
- [**Pydantic Validators**](https://docs.pydantic.dev/latest/concepts/validators/): Reference for the `before` validators used in the data models
- [**Plotly Gantt Charts**](https://plotly.com/python/gantt/): Timeline charts like the one the agent generates
- [**Building Effective Agents**](https://www.anthropic.com/engineering/building-effective-agents): A practical article on when to use workflows and when to use agents